# Experiment 7: Resize + Normalisasi Min-Max + CLAHE HSV + Gaussian Blur + Gamma Correction (balanced dataset)

Eksperimen ini berfokus pada alur preprocessing lengkap:
1. **Resize** ke 256x256 piksel (otomatis saat pemuatan dataset).
2. **Normalisasi Min-Max** untuk meratakan kontras global citra ke rentang [0, 255].
3. **CLAHE pada HSV** (kontras lokal ditingkatkan pada S & V) untuk memperjelas batas awan.
4. **Gaussian Blur (kernel 5x5, sigma=1.0)** untuk meredam noise sensor dan menekan amplifikasi noise akibat operasi kontras.
5. **Gamma Correction (gamma=1.2)** untuk meningkatkan detail pada area midtone/shadows awan.
6. **RGB/Color-based Extraction & Texture Extraction** (HSV, NRBR, LBP, dan GLCM) untuk melatih model klasifikasi.

In [1]:
import sys
sys.path.append('..')  # supaya src/ bisa diimport dari notebooks/

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import graycomatrix, graycoprops
from src.loader import get_dataset_root, load_dataset
from src.image_processing import *

EXPERIMENT_NAME = "experiment7"

c:\Users\Mahesa\miniconda3\envs\imgproc\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


CuPy tidak aktif atau tidak berfungsi (fallback ke CPU/NumPy): Failed to find CUDA headers. Please install CUDA toolkit headers (e.g., pip install cupy-cuda12x[ctk]) or specify CUDA_PATH environment variable.


## Data Loading (Balanced to Max 1250 per Class)

In [2]:
DATASET_ROOT = get_dataset_root()
# Muat gambar warna (color=True) agar ekstraksi HSV dan NRBR bisa dilakukan
# Default max_per_class=1250 dan target_size=(256, 256) otomatis digunakan
images, labels, filenames = load_dataset(DATASET_ROOT, target_size=(256, 256), color=True)
print(f"Dataset loaded: {len(images)} gambar warna, {len(set(labels))} kelas")

Environment: Lokal
DATASET_ROOT dari .env: D:\INFORMATICS\SEMESTER 4\IMAGE PROCESSING\Praktikum\Project\GCD


Loading altocumulus:   0%|          | 0/1250 [00:00<?, ?it/s]

Loading altocumulus:   1%|          | 13/1250 [00:00<00:10, 119.81it/s]

Loading altocumulus:   2%|▏         | 29/1250 [00:00<00:08, 140.95it/s]

Loading altocumulus:   4%|▎         | 44/1250 [00:00<00:08, 144.97it/s]

...

Loading stratocumulus:  85%|████████▍ | 1062/1250 [00:02<00:00, 465.68it/s]

Loading stratocumulus:  89%|████████▊ | 1109/1250 [00:02<00:00, 462.25it/s]

Loading stratocumulus:  93%|█████████▎| 1159/1250 [00:02<00:00, 470.16it/s]

Loading stratocumulus:  97%|█████████▋| 1213/1250 [00:02<00:00, 485.58it/s]

Dataset loaded: 8455 gambar, 7 kelas, ukuran=(256, 256)
Dataset loaded: 8455 gambar warna, 7 kelas


## Preprocessing Pipeline (Normalisasi -> CLAHE HSV -> Gaussian Blur -> Gamma Correction)

In [3]:
# ================================================================
# EDIT BAGIAN INI untuk mengubah pipeline preprocessing
# Tambah, hapus, atau ganti fungsi sesuai experiment
# ================================================================

PIPELINE = [
    lambda img: normalize_minmax(img),
    lambda img: clahe_hsv(img, tile_size=8, clip_limit=2.0),
    lambda img: gaussian_filter(img, kernel_size=5, sigma=1.0),
    lambda img: gamma_correction(img, gamma=1.2),
]

# ================================================================
# Jangan edit di bawah ini
# ================================================================

def apply_pipeline(image: np.ndarray, pipeline: list) -> np.ndarray:
    for fn in pipeline:
        image = fn(image)
    return image


In [4]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def process_one(img):
    return apply_pipeline(img, PIPELINE)

print("Menjalankan Preprocessing Pipeline...")
if USING_GPU:
    print("Menggunakan ThreadPoolExecutor (GPU aktif)...")
    with ThreadPoolExecutor(max_workers=4) as executor:
        images_preprocessed_color = list(tqdm(
            executor.map(process_one, images),
            total=len(images),
            desc="Preprocessing"
        ))
else:
    print("Menggunakan Pemrosesan Sekuensial CPU...")
    images_preprocessed_color = [process_one(img) for img in tqdm(images, desc="Preprocessing")]
    
images_preprocessed_color = np.array(images_preprocessed_color)

Menjalankan Preprocessing Pipeline...
Menggunakan Pemrosesan Sekuensial CPU...


Preprocessing:   0%|          | 0/8455 [00:00<?, ?it/s]

Preprocessing:   0%|          | 1/8455 [00:00<19:18,  7.30it/s]

Preprocessing:   0%|          | 2/8455 [00:00<16:18,  8.64it/s]

Preprocessing:   0%|          | 3/8455 [00:00<17:49,  7.91it/s]

...

Preprocessing: 100%|█████████▉| 8444/8455 [08:04<00:00, 21.79it/s]

Preprocessing: 100%|█████████▉| 8447/8455 [08:04<00:00, 21.74it/s]

Preprocessing: 100%|█████████▉| 8450/8455 [08:04<00:00, 21.29it/s]

Preprocessing: 100%|█████████▉| 8453/8455 [08:04<00:00, 21.52it/s]

Preprocessing: 100%|██████████| 8455/8455 [08:04<00:00, 17.45it/s]

## Visualisasi Preprocessing Bertahap

Melihat efek dari setiap tahapan preprocessing secara bertahap.

In [ ]:
# Visualisasi Preprocessing Bertahap secara otomatis untuk semua tahapan
visualize_pipeline_steps(images, labels, PIPELINE)


In [5]:
# Konversi hasil preprocessing ke grayscale khusus untuk ekstraksi fitur GLCM & LBP
print("Mengonversi gambar preprocessed ke grayscale...")
images_preprocessed = np.array([
    cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in tqdm(images_preprocessed_color, desc="Converting to grayscale")
])

Mengonversi gambar preprocessed ke grayscale...


Converting to grayscale:   0%|          | 0/8455 [00:00<?, ?it/s]

Converting to grayscale:   7%|▋         | 616/8455 [00:00<00:01, 6055.72it/s]

Converting to grayscale:  19%|█▉        | 1604/8455 [00:00<00:00, 8237.70it/s]

Converting to grayscale:  35%|███▍      | 2925/8455 [00:00<00:00, 10462.73it/s]

Converting to grayscale:  50%|█████     | 4262/8455 [00:00<00:00, 11530.43it/s]

Converting to grayscale:  66%|██████▌   | 5595/8455 [00:00<00:00, 12169.22it/s]

Converting to grayscale:  81%|████████  | 6813/8455 [00:00<00:00, 11850.82it/s]

Converting to grayscale:  95%|█████████▌| 8063/8455 [00:00<00:00, 12034.72it/s]

Converting to grayscale: 100%|██████████| 8455/8455 [00:00<00:00, 11384.72it/s]

## Feature Extraction - Color & Texture

Kita mengekstrak:
- **HSV Color Stats & Histograms**
- **NRBR Color Stats & Histograms**
- **LBP Texture Histograms**
- **GLCM Texture Features**

In [6]:
# Ekstrasi Fitur Warna HSV
print("Mengekstrak fitur HSV...")
h_hists, s_hists, v_hists, \
h_means, h_stds, h_skews, h_kurts, \
s_means, s_stds, s_skews, s_kurts, \
v_means, v_stds, v_skews, v_kurts = extract_hsv_features(images_preprocessed_color)

# Ekstraksi Fitur Warna NRBR (Normalized Red-Blue Ratio)
print("Mengekstrak fitur NRBR...")
nrbr_hists, nrbr_means, nrbr_stds, nrbr_skews, nrbr_kurts = extract_nrbr_features(images_preprocessed_color)

# Ekstraksi Fitur Tekstur LBP (Local Binary Pattern)
print("Mengekstrak fitur LBP...")
lbp_hists = extract_lbp_features(images_preprocessed, bins=8)

Mengekstrak fitur HSV...


Mengekstrak fitur NRBR...


Mengekstrak fitur LBP...


In [7]:
# Ekstraksi Fitur Tekstur GLCM
def glcm(image, angle):
    if angle == 0:
        angles = [0]
    elif angle == 45:
        angles = [np.pi / 4]
    elif angle == 90:
        angles = [np.pi / 2]
    elif angle == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Sudut tidak valid. Gunakan 0, 45, 90, atau 135.")
    return graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)

def correlation(matriks):
    return graycoprops(matriks, 'correlation')[0, 0]
def dissimilarity(matriks):
    return graycoprops(matriks, 'dissimilarity')[0, 0]
def homogenity(matriks):
    return graycoprops(matriks, 'homogeneity')[0, 0]
def contrast(matriks):
    return graycoprops(matriks, 'contrast')[0, 0]
def ASM(matriks):
    return graycoprops(matriks, 'ASM')[0, 0]
def energy(matriks):
    return graycoprops(matriks, 'energy')[0, 0]
def entropyGlcm(matriks):
    return entropy(matriks.ravel())

Kontras0, Kontras45, Kontras90, Kontras135 = [], [], [], []
dissimilarity0, dissimilarity45, dissimilarity90, dissimilarity135 = [], [], [], []
homogenity0, homogenity45, homogenity90, homogenity135 = [], [], [], []
entropy0, entropy45, entropy90, entropy135 = [], [], [], []
ASM0, ASM45, ASM90, ASM135 = [], [], [], []
energy0, energy45, energy90, energy135 = [], [], [], []
correlation0, correlation45, correlation90, correlation135 = [], [], [], []

for i in tqdm(range(len(images_preprocessed)), desc="Extracting GLCM features"):
    g0 = glcm(images_preprocessed[i], 0)
    g45 = glcm(images_preprocessed[i], 45)
    g90 = glcm(images_preprocessed[i], 90)
    g135 = glcm(images_preprocessed[i], 135)
    
    Kontras0.append(contrast(g0))
    Kontras45.append(contrast(g45))
    Kontras90.append(contrast(g90))
    Kontras135.append(contrast(g135))
    
    dissimilarity0.append(dissimilarity(g0))
    dissimilarity45.append(dissimilarity(g45))
    dissimilarity90.append(dissimilarity(g90))
    dissimilarity135.append(dissimilarity(g135))
    
    homogenity0.append(homogenity(g0))
    homogenity45.append(homogenity(g45))
    homogenity90.append(homogenity(g90))
    homogenity135.append(homogenity(g135))
    
    entropy0.append(entropyGlcm(g0))
    entropy45.append(entropyGlcm(g45))
    entropy90.append(entropyGlcm(g90))
    entropy135.append(entropyGlcm(g135))
    
    ASM0.append(ASM(g0))
    ASM45.append(ASM(g45))
    ASM90.append(ASM(g90))
    ASM135.append(ASM(g135))
    
    energy0.append(energy(g0))
    energy45.append(energy(g45))
    energy90.append(energy(g90))
    energy135.append(energy(g135))
    
    correlation0.append(correlation(g0))
    correlation45.append(correlation(g45))
    correlation90.append(correlation(g90))
    correlation135.append(correlation(g135))

Extracting GLCM features:   0%|          | 0/8455 [00:00<?, ?it/s]

Extracting GLCM features:   0%|          | 4/8455 [00:00<03:36, 39.07it/s]

Extracting GLCM features:   0%|          | 13/8455 [00:00<02:06, 66.64it/s]

Extracting GLCM features:   0%|          | 23/8455 [00:00<01:48, 77.47it/s]

Extracting GLCM features:   0%|          | 31/8455 [00:00<01:56, 72.12it/s]

...

Extracting GLCM features: 100%|█████████▉| 8440/8455 [02:11<00:00, 34.54it/s]

Extracting GLCM features: 100%|█████████▉| 8445/8455 [02:11<00:00, 37.99it/s]

Extracting GLCM features: 100%|█████████▉| 8450/8455 [02:11<00:00, 39.65it/s]

Extracting GLCM features: 100%|██████████| 8455/8455 [02:11<00:00, 40.35it/s]

Extracting GLCM features: 100%|██████████| 8455/8455 [02:11<00:00, 64.10it/s]

In [8]:
# Gabungkan semua fitur ke dalam satu DataFrame
dataTable = {
    'Filename': filenames,
    'Label': labels,
    # GLCM
    'Contrast0': Kontras0, 'Contrast45': Kontras45, 'Contrast90': Kontras90, 'Contrast135': Kontras135,
    'Homogeneity0': homogenity0, 'Homogeneity45': homogenity45, 'Homogeneity90': homogenity90, 'Homogeneity135': homogenity135,
    'Dissimilarity0': dissimilarity0, 'Dissimilarity45': dissimilarity45, 'Dissimilarity90': dissimilarity90, 'Dissimilarity135': dissimilarity135,
    'Entropy0': entropy0, 'Entropy45': entropy45, 'Entropy90': entropy90, 'Entropy135': entropy135,
    'ASM0': ASM0, 'ASM45': ASM45, 'ASM90': ASM90, 'ASM135': ASM135,
    'Energy0': energy0, 'Energy45': energy45, 'Energy90': energy90, 'Energy135': energy135,
    'Correlation0': correlation0, 'Correlation45': correlation45, 'Correlation90': correlation90, 'Correlation135': correlation135,
    # HSV Stats
    'HSV_H_Mean': h_means, 'HSV_H_Std': h_stds, 'HSV_H_Skew': h_skews, 'HSV_H_Kurt': h_kurts,
    'HSV_S_Mean': s_means, 'HSV_S_Std': s_stds, 'HSV_S_Skew': s_skews, 'HSV_S_Kurt': s_kurts,
    'HSV_V_Mean': v_means, 'HSV_V_Std': v_stds, 'HSV_V_Skew': v_skews, 'HSV_V_Kurt': v_kurts,
    # NRBR Stats
    'NRBR_Mean': nrbr_means, 'NRBR_Std': nrbr_stds, 'NRBR_Skew': nrbr_skews, 'NRBR_Kurt': nrbr_kurts,
}

# Tambahkan HSV Histograms
for b in range(16):
    dataTable[f'HSV_H_Hist_Bin_{b}'] = h_hists[:, b]
for b in range(8):
    dataTable[f'HSV_S_Hist_Bin_{b}'] = s_hists[:, b]
for b in range(8):
    dataTable[f'HSV_V_Hist_Bin_{b}'] = v_hists[:, b]

# Tambahkan NRBR Histograms
for b in range(16):
    dataTable[f'NRBR_Hist_Bin_{b}'] = nrbr_hists[:, b]

# Tambahkan LBP Histograms
for b in range(8):
    dataTable[f'LBP_Hist_Bin_{b}'] = lbp_hists[:, b]

df = pd.DataFrame(dataTable)
csv_path = f'hasil_ekstraksi_{EXPERIMENT_NAME}.csv'
df.to_csv(csv_path, index=False)
print(f'Fitur gabungan berhasil disimpan ke {csv_path}')
df.head()

Fitur gabungan berhasil disimpan ke hasil_ekstraksi_experiment7.csv


,Filename,Label,Contrast0,Contrast45,Contrast90,Contrast135,Homogeneity0,Homogeneity45,Homogeneity90,Homogeneity135,...,NRBR_Hist_Bin_14,NRBR_Hist_Bin_15,LBP_Hist_Bin_0,LBP_Hist_Bin_1,LBP_Hist_Bin_2,LBP_Hist_Bin_3,LBP_Hist_Bin_4,LBP_Hist_Bin_5,LBP_Hist_Bin_6,LBP_Hist_Bin_7
0,2_altocumulus_001371.jpg,altocumulus,40.775245,68.317693,53.069409,109.373610,0.444436,0.356638,0.414099,0.350389,...,0.0,0.0,0.005022,0.002074,0.000188,0.003516,0.002617,0.000249,0.003530,0.014054
1,2_altocumulus_000676.jpg,altocumulus,57.111765,131.910281,86.972273,147.136332,0.188943,0.126553,0.151879,0.121817,...,0.0,0.0,0.009438,0.003016,0.000186,0.003321,0.003275,0.000196,0.003170,0.008648
2,2_altocumulus_001452.jpg,altocumulus,44.792142,99.253749,58.116774,102.821561,0.227448,0.152434,0.205733,0.168055,...,0.0,0.0,0.008182,0.003631,0.000098,0.003468,0.003144,0.000110,0.003613,0.009005
3,2_altocumulus_000336.jpg,altocumulus,70.553171,148.056471,104.989844,190.781561,0.359902,0.279251,0.307238,0.252632,...,0.0,0.0,0.007486,0.002597,0.000213,0.003524,0.002735,0.000217,0.002612,0.011866
4,2_altocumulus_000201.jpg,altocumulus,34.986949,107.669988,93.377374,143.817178,0.290522,0.198616,0.221322,0.181443,...,0.0,0.0,0.010111,0.001965,0.000147,0.002636,0.002906,0.000167,0.003104,0.010213


## Train-Test Split & Normalization

In [9]:
# Pisahkan fitur dan label
X = df.drop(columns=['Filename', 'Label']).values
y = df['Label'].values

# Split 80/20 secara acak dan terstratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Z-score Normalization (dengan epsilon 1e-8 untuk menghindari pembagian dengan nol)
mean_train = X_train.mean(axis=0)
std_train = X_train.std(axis=0)
X_train_norm = (X_train - mean_train) / (std_train + 1e-8)
X_test_norm = (X_test - mean_train) / (std_train + 1e-8)
print(f"Train shape: {X_train_norm.shape}")
print(f"Test shape: {X_test_norm.shape}")

Train shape: (6764, 100)
Test shape: (1691, 100)


## Classifiers Training & Evaluation

In [10]:
def evaluate_model(clf, X_tr, y_tr, X_ts, y_ts, name):
    clf.fit(X_tr, y_tr)
    y_pred_tr = clf.predict(X_tr)
    y_pred_ts = clf.predict(X_ts)
    
    print(f"\n====== {name} - Training Set ======")
    print(classification_report(y_tr, y_pred_tr, zero_division=0))
    print(f"Accuracy: {accuracy_score(y_tr, y_pred_tr):.4f}")
    
    print(f"\n====== {name} - Testing Set ======")
    print(classification_report(y_ts, y_pred_ts, zero_division=0))
    print(confusion_matrix(y_ts, y_pred_ts))
    print(f"Accuracy: {accuracy_score(y_ts, y_pred_ts):.4f}")
    return y_pred_ts

# Definisikan model dengan class_weight='balanced' untuk penanganan optimal jika ada minor imbalans
rf = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
svm = SVC(kernel='rbf', C=2.0, random_state=42, class_weight='balanced')
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')

y_pred_rf_test = evaluate_model(rf, X_train_norm, y_train, X_test_norm, y_test, "Random Forest")
y_pred_svm_test = evaluate_model(svm, X_train_norm, y_train, X_test_norm, y_test, "SVM")
y_pred_knn_test = evaluate_model(knn, X_train_norm, y_train, X_test_norm, y_test, "KNN")


====== Random Forest - Training Set ======
               precision    recall  f1-score   support

  altocumulus       1.00      1.00      1.00      1000
       cirrus       1.00      1.00      1.00      1000
     clearsky       1.00      1.00      1.00      1000
 cumulonimbus       1.00      1.00      1.00      1000
      cumulus       1.00      1.00      1.00      1000
        mixed       1.00      1.00      1.00       764
stratocumulus       1.00      1.00      1.00      1000

     accuracy                           1.00      6764
    macro avg       1.00      1.00      1.00      6764
 weighted avg       1.00      1.00      1.00      6764

Accuracy: 1.0000

====== Random Forest - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.80      0.87      0.84       250
       cirrus       0.75      0.69      0.72       250
     clearsky       0.93      0.97      0.95       250
 cumulonimbus       0.70      0.74      0.72       250
      cumulu


====== SVM - Training Set ======
               precision    recall  f1-score   support

  altocumulus       0.89      0.87      0.88      1000
       cirrus       0.81      0.85      0.83      1000
     clearsky       0.96      0.98      0.97      1000
 cumulonimbus       0.87      0.83      0.85      1000
      cumulus       0.92      0.92      0.92      1000
        mixed       0.80      0.80      0.80       764
stratocumulus       0.81      0.81      0.81      1000

     accuracy                           0.87      6764
    macro avg       0.87      0.87      0.87      6764
 weighted avg       0.87      0.87      0.87      6764

Accuracy: 0.8674

====== SVM - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.86      0.88      0.87       250
       cirrus       0.78      0.76      0.77       250
     clearsky       0.94      0.97      0.95       250
 cumulonimbus       0.70      0.80      0.75       250
      cumulus       0.89      0.


====== KNN - Training Set ======
               precision    recall  f1-score   support

  altocumulus       1.00      1.00      1.00      1000
       cirrus       1.00      1.00      1.00      1000
     clearsky       1.00      1.00      1.00      1000
 cumulonimbus       1.00      1.00      1.00      1000
      cumulus       1.00      1.00      1.00      1000
        mixed       1.00      1.00      1.00       764
stratocumulus       1.00      1.00      1.00      1000

     accuracy                           1.00      6764
    macro avg       1.00      1.00      1.00      6764
 weighted avg       1.00      1.00      1.00      6764

Accuracy: 1.0000

====== KNN - Testing Set ======
               precision    recall  f1-score   support

  altocumulus       0.80      0.79      0.79       250
       cirrus       0.72      0.66      0.69       250
     clearsky       0.89      0.94      0.91       250
 cumulonimbus       0.65      0.73      0.69       250
      cumulus       0.84      0.

## Save Experiment Metrics & Auto-update README

In [11]:
from pathlib import Path
METRICS_PATH = Path('../results/metrics.csv')
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

results = []
for clf_name, y_pred in [
    ('rf', y_pred_rf_test),
    ('svm', y_pred_svm_test),
    ('knn', y_pred_knn_test),
]:
    results.append({
        'experiment_name': EXPERIMENT_NAME,
        'classifier': clf_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    })

new_rows = pd.DataFrame(results)

if METRICS_PATH.exists() and METRICS_PATH.stat().st_size > 0:
    all_metrics = pd.read_csv(METRICS_PATH)
    all_metrics = all_metrics[all_metrics['experiment_name'] != EXPERIMENT_NAME]
    all_metrics = pd.concat([all_metrics, new_rows], ignore_index=True)
else:
    all_metrics = new_rows

all_metrics.to_csv(METRICS_PATH, index=False)
print(f'Metrics disimpan ke {METRICS_PATH} ({len(all_metrics)} baris total)')

# Jalankan script generate_metrics_table.py untuk memperbarui tabel hasil di README.md secara otomatis
import subprocess
try:
    subprocess.run(["python", "../src/generate_metrics_table.py"], check=True)
    print("README.md berhasil diperbarui dengan tabel hasil terbaru!")
except Exception as e:
    print(f"Peringatan: Gagal meng-update README.md: {e}")

all_metrics

Metrics disimpan ke ..\results\metrics.csv (24 baris total)


README.md berhasil diperbarui dengan tabel hasil terbaru!


,experiment_name,classifier,accuracy,precision,recall,f1
0,experiment1,rf,0.821316,0.821657,0.821316,0.821135
1,experiment1,svm,0.787632,0.802315,0.787632,0.792130
2,experiment1,knn,0.782895,0.776283,0.782895,0.776533
3,experiment2,rf,0.829737,0.829593,0.829737,0.829322
4,experiment2,svm,0.793158,0.802144,0.793158,0.796048
5,experiment2,knn,0.780263,0.773187,0.780263,0.774127
6,experiment3,rf,0.655789,0.654693,0.655789,0.653106
7,experiment3,svm,0.609211,0.621162,0.609211,0.603450
8,experiment3,knn,0.621053,0.615394,0.621053,0.613233
9,experiment4,rf,0.832105,0.831199,0.832105,0.831413


## Analisis Eksperimen 13 (Sequential: Experiment 7)

Silakan berikan ulasan analisis Anda di sini.